In [ ]:
import ezc3d
import numpy as np
import os
from scipy.interpolate import interp1d
from scipy.signal import find_peaks
from pathlib import Path

In [ ]:
# === Ścieżki plików ===
root = Path(r"E:/marcel.furs/Documents/Data_Run_Walk/Data_Run_Walk")

train_subj = ["AJ026","BD004", "BL025", "CF027", "CL007", "DA013", "DC005", "GF022", "GM001", "GQ016", "HN021", "HS018", "JS009", "LA014", "LD002", "LF003", "LL011", "LM012", "MR015", "OB010"]  # 20 osób do trenowania
test_subj  = ["PC008", "PQ006", "RC020", "RC023", "RV028", "SA017", "SM019", "TB030", "TK029","YX024"]  # 10 osób do testowania (chodzenie )

# 3) Parametry katalogów
sessions      = ["Session1","Session2"]
walk_folder   = "Overground_Walk"
walk_speeds   = ["Walk_Slow","Walk_Fast","Walk_Comfortable"]
run_folder    = "Overground_Run"
run_speed     = "Run_Comfortable"
postproc      = "Post_Process"

def gather_walk(person, session):
    """Zwraca listę wszystkich plików .c3d z chodzeniem dla danej osoby i sesji."""
    files = []
    base = root / person / session / walk_folder
    for speed in walk_speeds:
        pp = base / speed / postproc
        i = 1
        while True:
            f = pp / f"{speed}{i}.c3d"
            if not f.exists():
                break
            files.append(str(f))
            i += 1
    return files

    
def gather_run(person, session):
    """Zwraca listę wszystkich plików .c3d z biegiem dla danej osoby i sesji."""
    files = []
    pp = root / person / session / run_folder / run_speed / postproc
    i = 1
    while True:
        f = pp / f"{run_speed}{i}.c3d"
        if not f.exists():
            break
        files.append(str(f))
        i += 1
    return files

# 4) Zbiór plików
train_files   = []
for p in train_subj:
    for sess in sessions:
        train_files += gather_walk(p, sess)

test_files    = []
anomaly_files = []
for p in test_subj:
    for sess in sessions:
        test_files    += gather_walk(p, sess)
        anomaly_files += gather_run(p,  sess)

print("Train (walk)   files:", len(train_files))
print("Test  (walk)   files:", len(test_files))
print("Anomaly (run)  files:", len(anomaly_files))

In [ ]:
for p in train_subj:
    cnt = len(gather_walk(p,"Session1")) + len(gather_walk(p,"Session2"))
    print(p, "walk files:", cnt)


In [ ]:
def process_c3d(file_path, target_frames: int = 100, show_events: bool = False):
    """
    Wczytuje plik .c3d i zwraca listę cykli o kształcie (target_frames, 55, 3).
    """
    print(f"\nPrzetwarzanie: {file_path}")
    c3d = ezc3d.c3d(file_path)

    # ---------- 1. wybór markerów ----------------------------------
    wanted = [
        "LFHD","RFHD","LBHD","RBHD","GLAB","CLAV","STRN","C7","T2","T10","RBAK",
        "LMAS","RMAS","LASI","LPSI","LTHI","LTHAP","LTHAD","LKNE","LTIB",
        "LTIAP","LTIAD","LANK","LHEE","LTOE","LFMH","LVMH","RASI","RPSI",
        "RTHI","RTHAP","RTHAD","RKNE","RTIB","RTIAP","RTIAD","RANK","RHEE",
        "RTOE","RFMH","RVMH","LSHO","LUPA","LELB","LFRM","LWRA","LWRB","LFIN",
        "RSHO","RUPA","RELB","RFRM","RWRA","RWRB","RFIN"
    ]
    all_labels = c3d["parameters"]["POINT"]["LABELS"]["value"]
    idx = [all_labels.index(m) for m in wanted if m in all_labels]
    if len(idx) < len(wanted):                       # brak któregoś markera
        missing = set(wanted) - set(all_labels)
        print(f"⚠️  Pomijam plik – brak markerów: {sorted(missing)}")
        return []

    points = c3d["data"]["points"][:3, idx, :]       # (3,55,F)
    n_frames = points.shape[2]

    frame_rate  = c3d["parameters"]["POINT"]["RATE"]["value"][0]
    first_frame = c3d["header"]["points"]["first_frame"]

    # ---------- 2. zdarzenia --------------------------------------
    ev        = c3d["parameters"]["EVENT"]
    n_ev      = ev["USED"]["value"][0]
    labels    = [b.decode().strip() if isinstance(b, bytes) else b.strip()
                 for b in ev["LABELS"]["value"][:n_ev]]
    ctxs      = [b.decode().strip() if isinstance(b, bytes) else b.strip()
                 for b in ev["CONTEXTS"]["value"][:n_ev]]
    frame_times = ev["TIMES"]["value"][0][:n_ev].astype(int)   # wiersz 0
    sec_times   = ev["TIMES"]["value"][1][:n_ev]               # wiersz 1

    if show_events:
        print("== LISTA ZDARZEŃ ==")
        for i in range(n_ev):
            print(f"{i:2d}| frame={frame_times[i]:5d}  t={sec_times[i]:7.3f}s  "
                  f"label='{labels[i]}'  ctx='{ctxs[i]}'")

    def time2frame(i: int) -> int:
        """Zwróć numer klatki zdarzenia względem first_frame (0‑based)."""
        if sec_times[i] != 0:
            return int(sec_times[i] * frame_rate) - first_frame
        return frame_times[i] - first_frame

    def is_strike(side: str, lbl: str, ctx: str) -> bool:
        """rozpoznanie Foot Strike po LABEL + CONTEXT."""
        l = lbl.lower().replace(" ", "")
        c = ctx.lower().replace(" ", "")
        side = side.lower()
        return (
            ("strike" in l and side in (l + c)) or   # FootStrike, Left Foot Strike
            (l == f"{side[0]}hs")                    # LHS / RHS
        )

    lhs = [time2frame(i) for i in range(n_ev)
           if 0 <= time2frame(i) < n_frames and is_strike("left",  labels[i], ctxs[i])]
    if len(lhs) < 2:
        print("⚠️  Brak ≥2 zdarzeń LHS – pomijam plik.")
        return []

    strike_frames = sorted(lhs)
    print(f"  ➤ LHS wykryte: {strike_frames}")
    #rhs = [time2frame(i) for i in range(n_ev)
    #       if 0 <= time2frame(i) < n_frames and is_strike("right", labels[i], ctxs[i])]

    #strike_frames = lhs if len(lhs) >= 2 else rhs
    #side_used = "LHS" if len(lhs) >= 2 else ("RHS" if len(rhs) >= 2 else "marker‑LHEE")

    # ---------- 3. początek prawdziwych danych ---------------------
    real_start = int(np.argmax(~np.isnan(points).all(axis=0).all(axis=0)))
    print(f"📍 Realny początek danych: frame {real_start}")

    # ---------- 4. wycinanie i resampling --------------------------
    def take_cycle(a: int, b: int) -> list[np.ndarray]:
        """
        Zwraca listę 0‑, 1‑ lub 2‑elementową:
            • gdy nie da się użyć kroku – []  
            • gdy jest 1 sensowny pik      – [cykl]  
            • gdy są 2 piki                 – [cykl_lewy, cykl_prawy]
        """
        if b - a < 5:
            return []                       # zbyt krótki fragment

        seg = points[:, :, a:b]             # (3,55,L)
        if np.isnan(seg).any():
            return []                       # brakujące dane
        seg = np.transpose(seg, (2, 1, 0))  # (L,55,3)

        heel_idx = wanted.index("LHEE")
        z        = seg[:, heel_idx, 2]      # trajektoria Z pięty

        # ---------- wykrywanie swing‑pików -------------------------
        peaks, _ = find_peaks(z, prominence=40)   # próg 40 mm

        if len(peaks) == 0:
            return []                       # brak piku ⇒ odrzucamy

        if len(peaks) > 1:                  # ► dwa kroki w jednym oknie
            # dwa najwyższe piki
            top2 = peaks[np.argsort(z[peaks])[-2:]]
            top2.sort()
            # indeks minimalnej wysokości między pikami
            cut_local = np.argmin(z[top2[0]:top2[1] + 1])
            cut = a + top2[0] + cut_local   # liczony względem całej próbki

            # rekurencyjnie obie połówki
            return take_cycle(a, cut) + take_cycle(cut, b)

        # ---------- pojedynczy pik → zwykły cykl -------------------
        L = seg.shape[0]
        if L != target_frames:
            new_x = np.linspace(0, L - 1, target_frames)
            res   = np.empty((target_frames, seg.shape[1], 3), dtype=np.float32)
            for m in range(seg.shape[1]):
                for d in range(3):
                    f = interp1d(np.arange(L), seg[:, m, d], kind="linear")
                    res[:, m, d] = f(new_x)
            seg = res
        return [seg.astype(np.float32)]      # jeden poprawny cykl w liście

    # ---------- budowanie listy cykli ------------------------------
    cycles = []
    for s, e in zip(strike_frames[:-1], strike_frames[1:]):
        if s < real_start:
            continue
        cycles.extend(take_cycle(s, e))      # ❸ doklejamy listę

    print(f"  ➤ Zakwalifikowano {len(cycles)} cykli\n")
    return cycles

In [ ]:
# === Funkcja do przetworzenia wielu plików ===
def process_multiple_files(file_list, output_path):
    all_cycles = []
    for idx, file in enumerate(file_list, 1):
        print(f"[{idx}/{len(file_list)}] Przetwarzam: {file}")
        cycles = process_c3d(file)
        all_cycles.extend(cycles)
    all_cycles = np.array(all_cycles, dtype=np.float32)
    print(f"Kształt danych: {all_cycles.shape}")
    np.save(output_path, all_cycles)
    print(f"✅ Dane zapisane do: {output_path}")

In [ ]:
process_multiple_files(train_files, r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_train_multi.npy")
process_multiple_files(test_files, r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_test_multi.npy")
process_multiple_files(anomaly_files, r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_anomaly_multi.npy")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === Lista markerów użyta w preprocessing'u ===
selected_marker_names = [
    "LFHD", "RFHD", "LBHD", "RBHD", "GLAB", "CLAV", "STRN", "C7", "T2", "T10", "RBAK", "LMAS", "RMAS",
    "LASI", "LPSI", "LTHI", "LTHAP", "LTHAD", "LKNE", "LTIB", "LTIAP", "LTIAD", "LANK", "LHEE", "LTOE", "LFMH", "LVMH",
    "RASI", "RPSI", "RTHI", "RTHAP", "RTHAD", "RKNE", "RTIB", "RTIAP", "RTIAD", "RANK", "RHEE", "RTOE", "RFMH", "RVMH",
    "LSHO", "LUPA", "LELB", "LFRM", "LWRA", "LWRB", "LFIN",
    "RSHO", "RUPA", "RELB", "RFRM", "RWRA", "RWRB", "RFIN"
]

# === Ścieżka do danych ===
data_path = r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_train_multi_fixed_OSTATECZNY.npy"
data = np.load(data_path)
data_path1 = r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_test_multi_fixed_D.npy"
data1 = np.load(data_path1)
data_path2 = r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_anomaly_multi_fixed.npy"
data2 = np.load(data_path2)

print(f"Kształt danych: {data.shape}")  # (n_cycles, 100, n_markers, 3)
print(f"Kształt danych: {data1.shape}")  # (n_cycles, 100, n_markers, 3)
print(f"Kształt danych: {data2.shape}")  # (n_cycles, 100, n_markers, 3)

# === Parametry ===
n_samples = 851
marker_name = "LHEE" 
marker_idx = selected_marker_names.index(marker_name)

# === Wykres ===
plt.figure(figsize=(12, 6))
for i in range(min(n_samples, data.shape[0])):
    z_signal = data[i, :, marker_idx, 2]  # 2 to oś Z (wysokość)
    plt.plot(z_signal, label=f"Cycle {i+1}")

plt.title(f"Wysokość markera '{marker_name}' () w {n_samples} cyklach chodu")
plt.xlabel("Frame (resampled to 100)")
plt.ylabel(" position (normalized or mm)")
#plt.legend()
plt.grid()
plt.tight_layout()
plt.show()


In [ ]:
file_path = "E:/marcel.furs/Documents/Data_Run_Walk/Data_Run_Walk/AJ026/Session1/Overground_Walk/Walk_Slow/Post_Process/Walk_Slow1.c3d"
import ezc3d
# === Lista markerów do przetworzenia ===
selected_marker_names = [
    "LFHD", "RFHD", "LBHD", "RBHD", "GLAB", "CLAV", "STRN", "C7", "T2", "T10", "RBAK", "LMAS", "RMAS",
    "LASI", "LPSI", "LTHI", "LTHAP", "LTHAD", "LKNE", "LTIB", "LTIAP", "LTIAD", "LANK", "LHEE", "LTOE", "LFMH", "LVMH",
    "RASI", "RPSI", "RTHI", "RTHAP", "RTHAD", "RKNE", "RTIB", "RTIAP", "RTIAD", "RANK", "RHEE", "RTOE", "RFMH", "RVMH",
    "LSHO", "LUPA", "LELB", "LFRM", "LWRA", "LWRB", "LFIN",
    "RSHO", "RUPA", "RELB", "RFRM", "RWRA", "RWRB", "RFIN"
]

# === Wczytaj dane z pliku C3D ===
c3d = ezc3d.c3d(file_path)

labels = c3d['parameters']['POINT']['LABELS']['value']
label_to_index = {label: i for i, label in enumerate(labels)}
selected_indices = [label_to_index[name] for name in selected_marker_names if name in label_to_index]

if len(selected_indices) != len(selected_marker_names):
    print("⚠️ Niektóre wymagane markery nie zostały znalezione w pliku.")

points = c3d['data']['points'][:3, selected_indices, :]  # (X/Y/Z, markery, klatki)
n_frames = points.shape[2]

# === Wykres trajektorii markera LHEE ===
if "LHEE" in selected_marker_names:
    lhee_idx = selected_marker_names.index("LHEE")

    plt.figure(figsize=(12, 6))
    plt.plot(range(n_frames), points[2, lhee_idx, :], label='LHEE Y')
    plt.plot(range(n_frames), points[1, lhee_idx, :], label='LHEE X')
    plt.plot(range(n_frames), points[0, lhee_idx, :], label='LHEE Z')
    plt.legend()
    plt.grid()
    plt.title("Pozycja LHEE w trzech osiach")
    plt.xlabel("Klatka")
    plt.ylabel("Pozycja (mm)")
    plt.show()

    # OŚ X lewo prawo
    # OŚ Y przód tył
    # OŚ Z góra dół


In [ ]:
import ezc3d

c3d_path = r"E:/marcel.furs/Documents/Data_Run_Walk/Data_Run_Walk/AJ026/Session1/Overground_Walk/Walk_Comfortable/Post_Process/Walk_Comfortable1.c3d"
c3d = ezc3d.c3d(c3d_path)

labels = c3d['parameters']['POINT']['LABELS']['value']

# Wyświetlenie wszystkich markerów z indeksami
for i, label in enumerate(labels):
    print(f"{i}: {label}")

# Znalezienie indeksu dla LHEE
marker_name = 'LHEE'
if marker_name in labels:
    marker_idx = labels.index(marker_name)
    print(f"\nIndex dla {marker_name}: {marker_idx}")
else:
    print(f"{marker_name} nie znaleziony wśród markerów.")



In [ ]:
import ezc3d
import numpy as np
import pandas as pd

marker_name = "LFHD"

file_path = "E:/marcel.furs/Documents/Data_Run_Walk/Data_Run_Walk/AJ026/Session1/Overground_Walk/Walk_Slow/Post_Process/Walk_Slow4.c3d"

c3d = ezc3d.c3d(file_path)
labels = c3d['parameters']['POINT']['LABELS']['value']
marker_idx = labels.index(marker_name)


marker_data = c3d['data']['points'][:3, marker_idx, :].T  # → (frames, 3)

df = pd.DataFrame(marker_data, columns=["X", "Y", "Z"])
df["Frame"] = df.index
df["is_nan"] = df.isna().any(axis=1)


print(df.to_string(index=False))


In [ ]:
import ezc3d

def inspect_events(file_path):
    c3d = ezc3d.c3d(file_path)

    frame_rate = c3d['parameters']['POINT']['RATE']['value'][0]
    first_frame = c3d['header']['points']['first_frame']

    events = c3d['parameters']['EVENT']
    labels = events['LABELS']['value']
    contexts = events['CONTEXTS']['value']
    times_sec = events['TIMES']['value'][1]  # <== CZAS W SEKUNDACH

    print(f"\nEventy w pliku: {file_path}")
    print(f"Frame rate: {frame_rate} Hz, first_frame: {first_frame}")
    print("-" * 50)

    for i, t in enumerate(times_sec):
        frame_index = int(t * frame_rate) - first_frame
        print(f"{labels[i]} ({contexts[i]}): {t:.4f} s  →  klatka {frame_index}")


inspect_events("E:/marcel.furs/Documents/Data_Run_Walk/Data_Run_Walk/AJ026/Session1/Overground_Walk/Walk_Slow/Post_Process/Walk_Slow4.c3d")



In [ ]:
import numpy as np

data = np.load(r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_train.npy")   

print("Ksztalt:", data.shape)          # (n_cycles, 100, 55, 3)
print("Typ   :", data.dtype)           # float32
print("Min / max:", data.min(), data.max())

# 3. Jeden wybrany cykl / marker / oś
cycle_id   = 3        # pierwszy cykl
marker_id  = 24       # np. 24 = 'LHEE'
axis       = 2        # 0:X, 1:Y, 2:Z

traj = data[cycle_id, :, marker_id, axis]   # (100,) – przebieg Z dla danej pięty
print(traj[:10])        # pierwsze 10 klatek


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import os

# === Ścieżki z plikami ===
train_path = r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_train_multi_fixed_OSTATECZNY.npy"
test_path  = r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_test_multi_fixed_D.npy"
anom_path  = r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_anomaly_multi_fixed_D.npy"

# === Wczytanie ===
data_train = np.load(train_path)
#data_test  = np.load(test_path)
#data_anom  = np.load(anom_path)

# === Połączenie wszystkich cykli ===
data_all = np.concatenate([data_train], axis=0)   # (N, 100, 55, 3)

# === Lista markerów ===
selected_marker_names = [
    "LFHD","RFHD","LBHD","RBHD","GLAB","CLAV","STRN","C7","T2","T10","RBAK","LMAS","RMAS",
    "LASI","LPSI","LTHI","LTHAP","LTHAD","LKNE","LTIB","LTIAP","LTIAD","LANK","LHEE","LTOE","LFMH","LVMH",
    "RASI","RPSI","RTHI","RTHAP","RTHAD","RKNE","RTIB","RTIAP","RTIAD","RANK","RHEE","RTOE","RFMH","RVMH",
    "LSHO","LUPA","LELB","LFRM","LWRA","LWRB","LFIN",
    "RSHO","RUPA","RELB","RFRM","RWRA","RWRB","RFIN"
]

axes_labels = {0: "Z (medio-lateral)", 1: "X (forward / AP)", 2: "Y (vertical)"}

# === Przygotowanie PDF ===
out_path = os.path.expanduser("~/ZMIENIONEOSIE_OSTATECZNY.pdf")
with PdfPages(out_path) as pdf:
    for m_idx, m_name in enumerate(selected_marker_names):
        fig, axs = plt.subplots(3, 1, figsize=(11, 8.5), sharex=True)
        for ax_dim, ax in enumerate(axs):
            for cycle in range(data_all.shape[0]):
                sig = data_all[cycle, :, m_idx, ax_dim]
                ax.plot(sig, alpha=0.2, linewidth=0.6)
            ax.set_title(f"Marker: {m_name} • Oś: {axes_labels[ax_dim]}", fontsize=9)
            ax.set_ylabel("Pozycja (mm)")
            ax.grid(alpha=0.3)
        axs[-1].set_xlabel("Klatka (0–99)")
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"PDF zapisany: {out_path}")


In [ ]:
import numpy as np
import os

# ---------- parametry -------------------------------------------
progress_dim = 0          # 0 = X , 1 = Y  (zmień jeśli trzeba)
heel_name    = "LHEE"     # marker referencyjny
paths_in = {
    "train": r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_train_multi_fixed_OSTATE.npy",
    "test":  r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_test_multi_fixed.npy",
    "anom":  r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_anomaly_multi_fixed.npy",
}
# ----------------------------------------------------------------

selected_marker_names = [
    "LFHD","RFHD","LBHD","RBHD","GLAB","CLAV","STRN","C7","T2","T10","RBAK","LMAS","RMAS",
    "LASI","LPSI","LTHI","LTHAP","LTHAD","LKNE","LTIB","LTIAP","LTIAD","LANK","LHEE","LTOE",
    "LFMH","LVMH","RASI","RPSI","RTHI","RTHAP","RTHAD","RKNE","RTIB","RTIAP","RTIAD","RANK",
    "RHEE","RTOE","RFMH","RVMH","LSHO","LUPA","LELB","LFRM","LWRA","LWRB","LFIN",
    "RSHO","RUPA","RELB","RFRM","RWRA","RWRB","RFIN"
]
heel_idx = selected_marker_names.index(heel_name)

def fix_progress_axis(data):
    """
    data: ndarray (n_cycles, 100, 55, 3)
    zwraca kopię z wyrównanym kierunkiem progresji.
    """
    fixed = data.copy()
    # różnica: ostatnia - pierwsza klatka
    delta = fixed[:, -1, heel_idx, progress_dim] - fixed[:,  0, heel_idx, progress_dim]
    flip_mask = delta < 0                   # cykle do odwrócenia
    # odwróć znak na osi progresji dla WSZYSTKICH markerów w cyklu
    fixed[flip_mask, :, :, progress_dim] *= -1
    return fixed

for tag, in_path in paths_in.items():
    data = np.load(in_path)
    print(f"{tag}:  wczytano  {data.shape}")
    fixed = fix_progress_axis(data)
    out_path = os.path.splitext(in_path)[0] + "_G.npy"
    np.save(out_path, fixed)
    print(f"{tag}: zapisano  {out_path}")

In [ ]:
import numpy as np
import os

# ---------- parametry -------------------------------------------
progress_dim = 0          # oś Z (boczna)
marker_name  = "LSHO"     # marker referencyjny (lewy bark, najlepiej do kołysania)
paths_in = {
    "train": r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_train_multi_fixed.npy",
    "test":  r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_test_multi_fixed.npy",
    "anom":  r"E:/marcel.furs/Documents/Data_Run_Walk/dataset_anomaly_multi_fixed.npy",
}
# ----------------------------------------------------------------

selected_marker_names = [
    "LFHD","RFHD","LBHD","RBHD","GLAB","CLAV","STRN","C7","T2","T10","RBAK","LMAS","RMAS",
    "LASI","LPSI","LTHI","LTHAP","LTHAD","LKNE","LTIB","LTIAP","LTIAD","LANK","LHEE","LTOE",
    "LFMH","LVMH","RASI","RPSI","RTHI","RTHAP","RTHAD","RKNE","RTIB","RTIAP","RTIAD","RANK",
    "RHEE","RTOE","RFMH","RVMH","LSHO","LUPA","LELB","LFRM","LWRA","LWRB","LFIN",
    "RSHO","RUPA","RELB","RFRM","RWRA","RWRB","RFIN"
]
marker_idx = selected_marker_names.index(marker_name)
threshold = 200  # Próg oddzielający grupy
def fix_z_with_shift(data, lsho_idx, dim, threshold=200):
    """
    Flipuj i przesuwaj oś Z, aby grupy się pokrywały.
    """
    fixed = data.copy()
    mean_z = fixed[:, :, lsho_idx, dim].mean(axis=1)
    flip_mask = mean_z < threshold

    # Flipuj oś Z w cyklach poniżej progu
    fixed[flip_mask, :, :, dim] *= -1

    # Po flipie, policz nowe średnie grup
    mean_z_after = fixed[:, :, lsho_idx, dim].mean(axis=1)
    group_high = mean_z_after[~flip_mask].mean()
    group_low  = mean_z_after[flip_mask].mean()

    shift = group_high - group_low
    print(f"Przesunięcie dla cykli flipowanych: {shift:.2f} mm")

    # Przesuń tylko flipowane cykle w osi Z
    fixed[flip_mask, :, :, dim] += shift

    return fixed

# ----------------------------------------
# Pipeline
for tag, in_path in paths_in.items():
    data = np.load(in_path)
    print(f"{tag}:  wczytano  {data.shape}")
    fixed = fix_z_with_shift(data, marker_idx, progress_dim, threshold)
    out_path = os.path.splitext(in_path)[0] + "_OSTATECZNY.npy"
    np.save(out_path, fixed)
    print(f"{tag}: zapisano  {out_path}")
